# 04 · Segmentación (U-Net) y medición del ancho

## La cadena completa
`MobileNetV2` decide **si** hay grieta → `U-Net` dice **dónde** (píxel a píxel) → `ancho.py` mide **cuántos mm**.

## Qué necesitás
Pares imagen + máscara en `datos_seg/imagenes/` y `datos_seg/mascaras/` (mismo nombre de archivo, máscara blanca = grieta).
Surface Crack **no trae máscaras**. Conjuntos públicos de segmentación de grietas a buscar y verificar (licencia y disponibilidad):
DeepCrack, CrackForest, "Concrete Crack Segmentation". Mientras tanto, `ancho.segmentar_clasico` funciona **sin red**.

In [ ]:
# ===================== CELDA DE ARRANQUE: no hay que editar nada =====================
# Monta tu Drive, encuentra sola la carpeta del proyecto y Surface Crack, y deja todo listo.
from google.colab import drive
drive.mount("/content/drive")

import glob, os, sys
_raiz = "/content/drive/MyDrive"
_cands = sorted(glob.glob(f"{_raiz}/*/src/datos.py") + glob.glob(f"{_raiz}/*/*/src/datos.py"),
                key=os.path.getmtime, reverse=True)
if not _cands:
    raise FileNotFoundError("No encontré la carpeta del proyecto NUEVO en tu Drive (busco una carpeta que tenga src/datos.py). "
                            "Subí la carpeta del zip nuevo a tu Drive y volvé a ejecutar esta celda.")
PROYECTO = os.path.dirname(os.path.dirname(_cands[0]))
os.chdir(PROYECTO)
sys.path.insert(0, PROYECTO)
print("Proyecto:", PROYECTO)
!pip -q install scikit-image scikit-learn pandas matplotlib

from src import colab
RUTA_DATOS = colab.preparar(PROYECTO, _raiz)       # Surface Crack (zip -> disco local, o carpeta en Drive)

import json, shutil, zipfile
from pathlib import Path
import numpy as np, pandas as pd
from src import config, ancho, unet

import cv2
from tensorflow import keras

## Paso 1 · Entrenar la U-Net
Pérdida **BCE + Dice**: las grietas ocupan pocos píxeles; solo BCE aprendería a decir "nada" en todos lados, y Dice
penaliza no encontrarlas. Métricas: **IoU** y **Dice** (cuánto se solapa la máscara predicha con la real).

In [ ]:
ds, n = unet.dataset_segmentacion("datos_seg/imagenes", "datos_seg/mascaras", lote=8, entrenar=True)
n_val = max(1, int(0.2 * n) // 8)
ds_va, ds_tr = ds.take(n_val), ds.skip(n_val)     # ⚠ al repartir por lotes, agrupá por escena si hay varias fotos de un mismo muro
modelo_seg = unet.construir_unet(congelar_encoder=True)
modelo_seg.fit(ds_tr, validation_data=ds_va, epochs=30)
modelo_seg.save("models/unet.keras")

## Paso 2 · Medir el ancho: clásico vs. U-Net
Se mide con **los mismos** pasos (esqueleto + distancia al borde): solo cambia cómo se obtiene la máscara. Se compara
contra el ancho real si lo conocés (con una regla en la foto).

**Escala:** `mm_por_px` desde un objeto de tamaño conocido **en el mismo plano que la grieta**. Para fotos en ángulo,
`rectificar_con_referencia` (4 esquinas de una tarjeta u hoja carta) corrige la perspectiva **y** da la escala.

In [ ]:
foto = cv2.cvtColor(cv2.imread("data/campo/ejemplo.jpg"), cv2.COLOR_BGR2RGB)
mascara_clasica = ancho.segmentar_clasico(foto)
mascara_unet = unet.segmentar_con_unet(modelo_seg, foto)
MM_POR_PX = ancho.mm_por_px(85.6, 428)      # ej.: tarjeta de 85,6 mm que mide 428 px en la foto
for nombre, mascara in (("clásica", mascara_clasica), ("U-Net", mascara_unet)):
    print(nombre, ancho.medir_ancho(mascara, MM_POR_PX))
cv2.imwrite("results/medicion.png", ancho.dibujar_medicion(cv2.cvtColor(foto, cv2.COLOR_RGB2BGR), mascara_unet))